# 01 - Data Cleaning

**Objective**

The purpose of this notebook is to clean and prepare the streaming platform dataset for analysis. Data cleaning improves data quality by correcting inconsistencies, handling missing values, and standardizing formats to ensure accurate and reliable results.

**Dataset**

This dataset contains information about movies and TV shows available on major streaming platforms, including Netflix, Hulu, Prime Video, and Disney+.

**Data Cleaning Tasks**

The following steps are performed in this notebook:
- Load the dataset
- Inspect the data structure
- Identify and handle missing values
- Remove duplicate records
- Standardize text formatting
- Convert data types where necessary
- Verify the cleaned dataset
- Export the cleaned dataset for analysis

**Expected Outcome**

The final cleaned dataset will be used in the exploratory data analysis and platform comparison notebooks.

## Load Data

In [2]:
import pandas as pd
import numpy as np
from pathlib import Path

DATA_DIR = Path("/Users/veronicahuang/Desktop/streaming-bi-template/Data")

def load_csv(filename: str) -> pd.DataFrame:
    fp = DATA_DIR / filename

    if fp.exists():
        print(f"Loaded: {fp}")
        return pd.read_csv(fp)
    raise FileNotFoundError(
        f"Could not find {filename} in any of: {[str(d) for d in DATA_DIR]}"
    )

df_movie = load_csv("MoviesOnStreamingPlatforms.csv")
df_tv = load_csv("TVShowsOnStreamingPlatforms.csv")

print("df_movie shape:", df_movie.shape)
print("df_tv shape:", df_tv.shape)

df_movie.head()



Loaded: /Users/veronicahuang/Desktop/streaming-bi-template/Data/MoviesOnStreamingPlatforms.csv
Loaded: /Users/veronicahuang/Desktop/streaming-bi-template/Data/TVShowsOnStreamingPlatforms.csv
df_movie shape: (9515, 15)
df_tv shape: (5368, 15)


,ID,Title,Year,Age,Rotten Tomatoes,Netflix,Hulu,Prime Video,Disney+,Type,Genre,Country,Language,IMDb,IMDb_ID
0,1,The Irishman,2019,18+,98/100,1,0,0,0,0,"Biography, Crime, Drama",United States,"English, Italian, Latin, Spanish, German",7.8,tt1302006
1,2,Dangal,2016,7+,97/100,1,0,0,0,0,"Action, Biography, Drama","India, United States","Hindi, English",8.3,tt5074352
2,3,David Attenborough: A Life on Our Planet,2020,7+,95/100,1,0,0,0,0,"Documentary, Biography",United Kingdom,English,8.9,tt11989890
3,4,Lagaan: Once Upon a Time in India,2001,7+,94/100,1,0,0,0,0,"Drama, Musical, Sport","India, United States","Hindi, English",8.1,tt0169102
4,5,Roma,2018,18+,94/100,1,0,0,0,0,Drama,"Mexico, United States","Spanish, Mixtec, English, Japanese, German, Fr...",7.6,tt6155172


## Handle Missing Data

In [3]:
# 1) Quick missingness overview (share of missing values per column)
def missingness(df: pd.DataFrame) -> pd.Series:
    return df.isna().mean().sort_values(ascending=False)

print("Missingness (Movies):")
display(missingness(df_movie))

print("\nMissingness (TV Shows):")
display(missingness(df_tv))

Missingness (Movies):


Age                0.438991
Language           0.366684
Genre              0.366264
Country            0.328744
IMDb               0.292486
IMDb_ID            0.286705
Rotten Tomatoes    0.000736
ID                 0.000000
Title              0.000000
Year               0.000000
Netflix            0.000000
Hulu               0.000000
Prime Video        0.000000
Disney+            0.000000
Type               0.000000
dtype: float64


Missingness (TV Shows):


Language           0.619411
Genre              0.588301
IMDb_ID            0.499255
Country            0.497578
Age                0.396237
IMDb               0.162072
ID                 0.000000
Title              0.000000
Year               0.000000
Rotten Tomatoes    0.000000
Netflix            0.000000
Hulu               0.000000
Prime Video        0.000000
Disney+            0.000000
Type               0.000000
dtype: float64

In [4]:
# 2) Common parsing / conversion helpers
def add_rotten_tomatoes_score(df: pd.DataFrame, source_col: str = "Rotten Tomatoes") -> pd.DataFrame:
    """Convert '98/100' -> 98.0 (numeric). Leaves NaN if missing/unparseable."""
    if source_col in df.columns:
        df["RottenTomatoes_Score"] = pd.to_numeric(
            df[source_col].astype(str).str.extract(r"(\d+)")[0],
            errors="coerce"
        )
    return df

def add_imdb_score(df: pd.DataFrame, source_col: str = "IMDb") -> pd.DataFrame:
    """Convert '98/100' -> 98.0 (numeric). Leaves NaN if missing/unparseable."""
    if source_col in df.columns:
        df["IMDb_Score"] = pd.to_numeric(
            df[source_col].astype(str).str.extract(r"(\d+)")[0],
            errors="coerce"
        )
    return df

def add_age_min(df: pd.DataFrame, source_col: str = "Age") -> pd.DataFrame:
    """Convert '18+' -> 18, '7+' -> 7 (numeric). Leaves NaN if missing/unparseable."""
    if source_col in df.columns:
        df["Age_Min"] = pd.to_numeric(
            df[source_col].astype(str).str.extract(r"(\d+)")[0],
            errors="coerce"
        )
    return df

def standardize_title(df: pd.DataFrame, col: str = "Title") -> pd.DataFrame:
    if col in df.columns:
        df[col] = df[col].astype(str).str.strip() # Removes leading and trailing whitespace
    return df

def standardize_type(df: pd.DataFrame, col: str = "Type") -> pd.DataFrame:
    """The datasets encode Type as 0/1. We'll map to readable labels."""
    if col in df.columns:
        # Keep original for reference
        df["Type_raw"] = df[col]
        df[col] = df[col].map({0: "movie", 1: "tv_show"}).fillna(df[col].astype(str))
    return df

def coerce_year(df: pd.DataFrame, col: str = "Year") -> pd.DataFrame:
    if col in df.columns:
        df[col] = pd.to_numeric(df[col], errors="coerce")
    return df

def coerce_platform_flags(df: pd.DataFrame, platform_cols=None) -> pd.DataFrame:
    """Ensure platform columns are numeric 0/1 (safe for grouping & sums)."""
    if platform_cols is None:
        platform_cols = ["Netflix", "Hulu", "Prime Video", "Disney+"]
    for c in platform_cols:
        if c in df.columns:
            df[c] = pd.to_numeric(df[c], errors="coerce").fillna(0).astype(int)
    return df

# Apply conversions to BOTH datasets
for _df in (df_movie, df_tv):
    add_rotten_tomatoes_score(_df)
    add_age_min(_df)
    standardize_title(_df)
    standardize_type(_df)
    coerce_year(_df)
    coerce_platform_flags(_df)

df_movie[["Title", "Year", "Age", "Age_Min", "Rotten Tomatoes", "RottenTomatoes_Score", "Type"]].head()

,Title,Year,Age,Age_Min,Rotten Tomatoes,RottenTomatoes_Score,Type
0,The Irishman,2019,18+,18.0,98/100,98.0,movie
1,Dangal,2016,7+,7.0,97/100,97.0,movie
2,David Attenborough: A Life on Our Planet,2020,7+,7.0,95/100,95.0,movie
3,Lagaan: Once Upon a Time in India,2001,7+,7.0,94/100,94.0,movie
4,Roma,2018,18+,18.0,94/100,94.0,movie


In [5]:
# 3) Decide which columns are numeric vs categorical AFTER conversions
def get_num_cat_cols(df: pd.DataFrame):
    num_cols = df.select_dtypes(include=[np.number]).columns.tolist()
    cat_cols = [c for c in df.columns if c not in num_cols]
    return num_cols, cat_cols

num_cols_movie, cat_cols_movie = get_num_cat_cols(df_movie)
num_cols_tv, cat_cols_tv = get_num_cat_cols(df_tv)

print("Movies numeric columns:", num_cols_movie)
print("Movies categorical columns:", cat_cols_movie)

print("\nTV numeric columns:", num_cols_tv)
print("TV categorical columns:", cat_cols_tv)

Movies numeric columns: ['ID', 'Year', 'Netflix', 'Hulu', 'Prime Video', 'Disney+', 'IMDb', 'RottenTomatoes_Score', 'Age_Min', 'Type_raw']
Movies categorical columns: ['Title', 'Age', 'Rotten Tomatoes', 'Type', 'Genre', 'Country', 'Language', 'IMDb_ID']

TV numeric columns: ['ID', 'Year', 'Netflix', 'Hulu', 'Prime Video', 'Disney+', 'RottenTomatoes_Score', 'Age_Min', 'Type_raw']
TV categorical columns: ['Title', 'Age', 'Rotten Tomatoes', 'Type', 'Genre', 'Country', 'Language', 'IMDb', 'IMDb_ID']


In [6]:
# 4) Impute missing values
def impute_basic(df: pd.DataFrame) -> pd.DataFrame:
    num_cols, cat_cols = get_num_cat_cols(df)

    for c in num_cols:
        if df[c].isna().any():
            df[c] = df[c].fillna(df[c].median())

    for c in cat_cols:
        if df[c].isna().any():
            mode = df[c].mode(dropna=True)
            fill_val = mode.iloc[0] if len(mode) > 0 else "Unknown"
            df[c] = df[c].fillna(fill_val)

    return df

df_movie = impute_basic(df_movie)
df_tv = impute_basic(df_tv)

print("After imputation, missingness (Movies) top 5:")
display(missingness(df_movie).head())

print("\nAfter imputation, missingness (TV) top 5:")
display(missingness(df_tv).head())

After imputation, missingness (Movies) top 5:


ID                      0.0
Title                   0.0
Age_Min                 0.0
RottenTomatoes_Score    0.0
IMDb_ID                 0.0
dtype: float64


After imputation, missingness (TV) top 5:


ID                      0.0
Title                   0.0
Age_Min                 0.0
RottenTomatoes_Score    0.0
IMDb_ID                 0.0
dtype: float64

In [16]:
# 5) Optionally drop rows missing must-have columns
must_have = ["Title", "Year"]

df_movie_before = len(df_movie)
df_tv_before = len(df_tv)

df_movie = df_movie.dropna(subset=[c for c in must_have if c in df_movie.columns])
df_tv = df_tv.dropna(subset=[c for c in must_have if c in df_tv.columns])

print(f"Movies dropped (must-have): {df_movie_before - len(df_movie)}")
print(f"TV dropped (must-have): {df_tv_before - len(df_tv)}")


Movies dropped (must-have): 0
TV dropped (must-have): 0


In [17]:
# Check: remaining missing values (top 10 columns)
print("Remaining missing values (Movies):")
display(df_movie.isna().sum().sort_values(ascending=False).head(10))

print("\nRemaining missing values (TV):")
display(df_tv.isna().sum().sort_values(ascending=False).head(10))


Remaining missing values (Movies):


ID                      0
Genre                   0
Type_raw                0
Age_Min                 0
RottenTomatoes_Score    0
IMDb_ID                 0
IMDb                    0
Language                0
Country                 0
Type                    0
dtype: int64


Remaining missing values (TV):


ID                      0
Genre                   0
Type_raw                0
Age_Min                 0
RottenTomatoes_Score    0
IMDb_ID                 0
IMDb                    0
Language                0
Country                 0
Type                    0
dtype: int64

## Remove Duplicates

In [18]:
# Show duplicate records (entire-row duplicates)
dup_all_movies = df_movie[df_movie.duplicated(keep=False)]
dup_all_tv = df_tv[df_tv.duplicated(keep=False)]

print("Movies: # duplicated rows (entire row):", len(dup_all_movies))
print("TV: # duplicated rows (entire row):", len(dup_all_tv))

display(dup_all_movies.head(10))

Movies: # duplicated rows (entire row): 0
TV: # duplicated rows (entire row): 0


,ID,Title,Year,Age,Rotten Tomatoes,Netflix,Hulu,Prime Video,Disney+,Type,Genre,Country,Language,IMDb,IMDb_ID,RottenTomatoes_Score,Age_Min,Type_raw,Title_key


## Standardize Formats

In [19]:
# Strip whitespace and unify casing
for _df in (df_movie, df_tv):
    if "Title" in _df.columns:
        _df["Title_key"] = _df["Title"].astype(str).str.strip().str.upper()

df_movie[["Title", "Title_key"]].head()

,Title,Title_key
0,The Irishman,THE IRISHMAN
1,Dangal,DANGAL
2,David Attenborough: A Life on Our Planet,DAVID ATTENBOROUGH: A LIFE ON OUR PLANET
3,Lagaan: Once Upon a Time in India,LAGAAN: ONCE UPON A TIME IN INDIA
4,Roma,ROMA


In [20]:
# Preview cleaned Movies dataset
df_movie.head()

,ID,Title,Year,Age,Rotten Tomatoes,Netflix,Hulu,Prime Video,Disney+,Type,Genre,Country,Language,IMDb,IMDb_ID,RottenTomatoes_Score,Age_Min,Type_raw,Title_key
0,1,The Irishman,2019,18+,98/100,1,0,0,0,movie,"Biography, Crime, Drama",United States,"English, Italian, Latin, Spanish, German",7.8,tt1302006,98.0,18.0,0,THE IRISHMAN
1,2,Dangal,2016,7+,97/100,1,0,0,0,movie,"Action, Biography, Drama","India, United States","Hindi, English",8.3,tt5074352,97.0,7.0,0,DANGAL
2,3,David Attenborough: A Life on Our Planet,2020,7+,95/100,1,0,0,0,movie,"Documentary, Biography",United Kingdom,English,8.9,tt11989890,95.0,7.0,0,DAVID ATTENBOROUGH: A LIFE ON OUR PLANET
3,4,Lagaan: Once Upon a Time in India,2001,7+,94/100,1,0,0,0,movie,"Drama, Musical, Sport","India, United States","Hindi, English",8.1,tt0169102,94.0,7.0,0,LAGAAN: ONCE UPON A TIME IN INDIA
4,5,Roma,2018,18+,94/100,1,0,0,0,movie,Drama,"Mexico, United States","Spanish, Mixtec, English, Japanese, German, Fr...",7.6,tt6155172,94.0,18.0,0,ROMA


In [21]:
# Preview cleaned TV dataset
df_tv.head()

,ID,Title,Year,Age,Rotten Tomatoes,Netflix,Hulu,Prime Video,Disney+,Type,Genre,Country,Language,IMDb,IMDb_ID,RottenTomatoes_Score,Age_Min,Type_raw,Title_key
0,1,Breaking Bad,2008,18+,100/100,1,0,0,0,tv_show,"crime television series, drama television seri...",United States,"American English, Spanish language in the Amer...",10.0,tt0903747,100,18.0,1,BREAKING BAD
1,2,Stranger Things,2016,16+,96/100,1,0,0,0,tv_show,"science fiction television program, horror tel...",United States,English,9.0,tt4574334,96,16.0,1,STRANGER THINGS
2,3,Attack on Titan,2013,18+,95/100,1,1,0,0,tv_show,reality television,United States,English,9.0/10,tt0092400,95,18.0,1,ATTACK ON TITAN
3,4,Better Call Saul,2015,18+,94/100,1,0,0,0,tv_show,"legal drama, crime television series",United States,English,10.0,tt3032476,94,18.0,1,BETTER CALL SAUL
4,5,Dark,2017,16+,93/100,1,0,0,0,tv_show,"science fiction television program, drama tele...",Germany,German,10.0,tt5753856,93,16.0,1,DARK


In [22]:
# Quick sanity checks
def sanity_checks(df: pd.DataFrame, name: str):
    print(f"--- {name} ---")
    if "Year" in df.columns:
        print("Year min/max:", int(df["Year"].min()), int(df["Year"].max()))
    for c in ["Netflix", "Hulu", "Prime Video", "Disney+"]:
        if c in df.columns:
            print(f"{c} value counts:", df[c].value_counts().to_dict())
    if "RottenTomatoes_Score" in df.columns:
        print("RottenTomatoes_Score describe:")
        display(df["RottenTomatoes_Score"].describe())

sanity_checks(df_movie, "Movies")
sanity_checks(df_tv, "TV Shows")

--- Movies ---
Year min/max: 1914 2021
Netflix value counts: {0: 5820, 1: 3695}
Hulu value counts: {0: 8468, 1: 1047}
Prime Video value counts: {0: 5402, 1: 4113}
Disney+ value counts: {0: 8593, 1: 922}
RottenTomatoes_Score describe:


count    9515.000000
mean       53.543878
std        13.192884
min        10.000000
25%        44.000000
50%        52.000000
75%        62.000000
max        98.000000
Name: RottenTomatoes_Score, dtype: float64

--- TV Shows ---
Year min/max: 1904 2021
Netflix value counts: {0: 3397, 1: 1971}
Hulu value counts: {0: 3747, 1: 1621}
Prime Video value counts: {0: 3537, 1: 1831}
Disney+ value counts: {0: 5017, 1: 351}
RottenTomatoes_Score describe:


count    5368.000000
mean       47.220380
std        19.555753
min        10.000000
25%        36.000000
50%        48.000000
75%        60.000000
max       100.000000
Name: RottenTomatoes_Score, dtype: float64

In [23]:
df_movie.to_csv(DATA_DIR / "MoviesOnStreamingPlatforms_Cleaned.csv", index=False)
df_tv.to_csv(DATA_DIR / "TVShowsOnStreamingPlatforms_Cleaned.csv", index=False)